In [ ]:
import re

from datasets import load_dataset


dataset = load_dataset("openassistant-guanaco")

Repo card metadata block was not found. Setting CardData to empty.


In [2]:
def reverse_format(example):
    text = example["text"]
    # 使用正则提取 instruction 和 output
    match = re.match(r"### Human:(.*?)### Assistant:(.*)", text, re.DOTALL)
    if match:
        instruction = match.group(1).strip()
        output = match.group(2).strip()
        return {
            "input": output,        # 反向任务：用输出预测指令
            "output": instruction
        }
    else:
        return None

# 应用映射并过滤空值
processed_dataset = dataset["train"].map(reverse_format)
processed_dataset = processed_dataset.filter(lambda x: x is not None and x.get("input") and x.get("output"))

Map:   0%|          | 0/9846 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9846 [00:00<?, ? examples/s]

In [3]:
processed_dataset

Dataset({
    features: ['text', 'input', 'output'],
    num_rows: 9846
})

In [4]:
import json


with open("reversed_openassistant.json", "w", encoding="utf-8") as f:
    for ex in processed_dataset:
        json.dump({"input": ex["input"], "output": ex["output"]}, f, ensure_ascii=False)
        f.write("\n")